In [1]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "❌ No GPU — please re-check runtime type")

Wed Mar 11 16:10:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print("✅ GPU is ready!")
    print(result.stdout[:500])
else:
    print("❌ No GPU. Go to Runtime → Change runtime type → T4 GPU")

✅ GPU is ready!
Wed Mar 11 16:14:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|       


In [3]:
print("Installing... please wait 2-4 minutes ⏳")

!pip install unsloth --quiet
!pip install trl peft datasets transformers accelerate bitsandbytes --quiet

print("✅ All done! Ready for next step.")

Installing... please wait 2-4 minutes ⏳
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 95.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 87.8 MB/s eta 0:00:00
   ━━━━

In [5]:
import torch
import warnings
warnings.filterwarnings("ignore")

from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

print(f"✅ Everything imported!")
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0)}")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅ Everything imported!
GPU available: True
GPU name: Tesla T4


In [6]:
print("🔄 Downloading Llama 3.2... (this takes 3-5 minutes first time)")
print("   You'll see a progress bar — just wait for it to finish\n")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = 1024,
    dtype          = None,
    load_in_4bit   = True,
)

used = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\n✅ Model loaded!")
print(f"💾 GPU memory used: {used:.1f} GB out of {total:.1f} GB")

🔄 Downloading Llama 3.2... (this takes 3-5 minutes first time)
   You'll see a progress bar — just wait for it to finish

==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.



✅ Model loaded!
💾 GPU memory used: 2.4 GB out of 15.6 GB


In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r                          = 16,
    target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj"],
    lora_alpha                 = 16,
    lora_dropout               = 0.05,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 42,
)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ LoRA adapters attached!")
print(f"🧠 Total parameters    : {total/1e9:.2f} Billion")
print(f"✏️  Trainable parameters: {trainable/1e6:.2f} Million")
print(f"📊 We only train       : {100*trainable/total:.2f}% of the model")


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.3.4 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


✅ LoRA adapters attached!
🧠 Total parameters    : 1.87 Billion
✏️  Trainable parameters: 24.31 Million
📊 We only train       : 1.30% of the model


In [8]:
print("📥 Loading medical dataset...")

dataset = load_dataset(
    "medalpaca/medical_meadow_medical_flashcards",
    split="train"
)

print(f"✅ Dataset loaded!")
print(f"📊 Total samples: {len(dataset)}")
print(f"\n📋 Sample question:")
print(f"Q: {dataset[0]['input'][:200]}")
print(f"\n💊 Sample answer:")
print(f"A: {dataset[0]['output'][:200]}")

📥 Loading medical dataset...


README.md: 0.00B [00:00, ?B/s]

medical_meadow_wikidoc_medical_flashcard(…):   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/33955 [00:00<?, ? examples/s]

✅ Dataset loaded!
📊 Total samples: 33955

📋 Sample question:
Q: What is the relationship between very low Mg2+ levels, PTH levels, and Ca2+ levels?

💊 Sample answer:
A: Very low Mg2+ levels correspond to low PTH levels which in turn results in low Ca2+ levels.


In [9]:
EOS_TOKEN = tokenizer.eos_token

def format_prompt(examples):
    questions = examples["input"]
    answers   = examples["output"]
    texts = []
    for q, a in zip(questions, answers):
        text = f"""### Instruction:
You are a medical expert. Answer the following clinical question accurately.

### Question:
{q}

### Answer:
{a}""" + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

print("🔄 Formatting dataset...")

dataset = dataset.map(format_prompt, batched=True)

# Use 2000 samples for faster training
dataset = dataset.select(range(2000))

print(f"✅ Data formatted!")
print(f"📊 Training samples: {len(dataset)}")
print(f"\n📄 Preview of formatted sample:")
print(dataset[0]["text"][:400] + "...")

🔄 Formatting dataset...


Map:   0%|          | 0/33955 [00:00<?, ? examples/s]

✅ Data formatted!
📊 Training samples: 2000

📄 Preview of formatted sample:
### Instruction:
You are a medical expert. Answer the following clinical question accurately.

### Question:
What is the relationship between very low Mg2+ levels, PTH levels, and Ca2+ levels?

### Answer:
Very low Mg2+ levels correspond to low PTH levels which in turn results in low Ca2+ levels.<|eot_id|>...


In [10]:
trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    dataset_text_field = "text",
    max_seq_length     = 1024,
    args = TrainingArguments(
        output_dir                  = "./output",
        num_train_epochs            = 3,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        learning_rate               = 2e-4,
        warmup_ratio                = 0.05,
        weight_decay                = 0.01,
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        optim                       = "adamw_8bit",
        logging_steps               = 25,
        save_strategy               = "epoch",
        seed                        = 42,
        report_to                   = "none",
    ),
)

print("✅ Trainer is ready!")
print("📊 Training plan:")
print(f"   Samples   : 2000 medical Q&A pairs")
print(f"   Epochs    : 3  (reads data 3 times)")
print(f"   Est. time : ~15-20 minutes on T4")
print(f"\n🚀 Ready to train!")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2000 [00:00<?, ? examples/s]

✅ Trainer is ready!
📊 Training plan:
   Samples   : 2000 medical Q&A pairs
   Epochs    : 3  (reads data 3 times)
   Est. time : ~15-20 minutes on T4

🚀 Ready to train!


In [11]:
import time

print("🚀 Training started! Sit back and relax...")
print("   Watch the 'loss' number — it should go DOWN over time")
print("   Lower loss = model is learning better\n")
print("─" * 50)

start = time.time()

trainer_stats = trainer.train()

mins = int((time.time() - start) // 60)
secs = int((time.time() - start) % 60)

print("─" * 50)
print(f"✅ Training complete!")
print(f"⏱️  Time taken     : {mins} minutes {secs} seconds")
print(f"📉 Final loss     : {trainer_stats.training_loss:.4f}")
print(f"👣 Total steps    : {trainer_stats.global_step}")

🚀 Training started! Sit back and relax...
   Watch the 'loss' number — it should go DOWN over time
   Lower loss = model is learning better

──────────────────────────────────────────────────


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 3 | Total steps = 750
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss
25,1.571668
50,0.952294
75,0.846050
100,0.822935
125,0.806001
150,0.776312
175,0.736472
200,0.748402
225,0.716994
250,0.692132


──────────────────────────────────────────────────
✅ Training complete!
⏱️  Time taken     : 22 minutes 1 seconds
📉 Final loss     : 0.5737
👣 Total steps    : 750


In [12]:
print("💾 Saving your fine-tuned model...")

model.save_pretrained("./medical_lora_adapter")
tokenizer.save_pretrained("./medical_lora_adapter")

import os
files = os.listdir("./medical_lora_adapter")
size  = sum(os.path.getsize(f"./medical_lora_adapter/{f}")
            for f in files) / 1e6

print(f"✅ Model saved!")
print(f"📁 Files saved : {files}")
print(f"📦 Total size  : {size:.1f} MB")

💾 Saving your fine-tuned model...
✅ Model saved!
📁 Files saved : ['adapter_config.json', 'tokenizer.json', 'tokenizer_config.json', 'README.md', 'adapter_model.safetensors', 'chat_template.jinja']
📦 Total size  : 114.5 MB


In [13]:
FastLanguageModel.for_inference(model)

def ask(question):
    prompt = f"""### Instruction:
You are a medical expert. Answer the following clinical question accurately.

### Question:
{question}

### Answer:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = 300,
            temperature        = 0.3,
            do_sample          = True,
            pad_token_id       = tokenizer.eos_token_id,
            repetition_penalty = 1.1,
        )

    input_len = inputs["input_ids"].shape[1]
    response  = tokenizer.decode(
        outputs[0][input_len:],
        skip_special_tokens=True
    ).strip()

    print(f"❓ Question: {question}")
    print(f"\n💊 Answer:\n{response}")
    print("\n" + "─"*50)

# Test with 3 medical questions
ask("What are the symptoms of Type 2 Diabetes?")
ask("What is the mechanism of action of Aspirin?")
ask("What causes high blood pressure?")

--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, 

❓ Question: What are the symptoms of Type 2 Diabetes?

💊 Answer:
Type 2 diabetes is characterized by insulin resistance and impaired insulin secretion, which may present with polyuria, polydipsia, and unexplained weight loss.

──────────────────────────────────────────────────
❓ Question: What is the mechanism of action of Aspirin?

💊 Answer:
Aspirin exerts its effect by inhibiting COX-1, an enzyme that converts arachidonic acid to prostaglandins.

──────────────────────────────────────────────────
❓ Question: What causes high blood pressure?

💊 Answer:
High blood pressure is caused by narrowing or constriction of blood vessels, which increases vascular resistance and leads to increased afterload.

──────────────────────────────────────────────────


In [14]:
ask("What are the side effects of Metformin?")
ask("How is pneumonia diagnosed?")
ask("What is the difference between Type 1 and Type 2 Diabetes?")

❓ Question: What are the side effects of Metformin?

💊 Answer:
Metformin may cause GI distress and lactic acidosis (rare).

──────────────────────────────────────────────────
❓ Question: How is pneumonia diagnosed?

💊 Answer:
Pneumonia is diagnosed by culture and Gram stain.

──────────────────────────────────────────────────
❓ Question: What is the difference between Type 1 and Type 2 Diabetes?

💊 Answer:
Type 1 diabetes is an autoimmune condition in which the body's immune system attacks and destroys the insulin-producing cells (β cells) in the pancreas, while Type 2 diabetes is characterized by insulin resistance and impaired insulin secretion.

──────────────────────────────────────────────────
